# EDA: Quick Stats JSONL

Runs the same logic as `eda_jsonl.py` but with **periodic progress lines** (rows, elapsed time, rows/sec) so a multi‑GB file does not look “stuck”.

**Tip:** set `MAX_ROWS` (e.g. `500_000`) for a quick test, then set it to `None` for a full pass. In Jupyter, keep **Run All** off until paths look right.

CLI equivalent for long runs: `python3 eda_jsonl.py -i data/crops.jsonl --progress-every 250000`

In [ ]:
from pathlib import Path
import time

from eda_jsonl import run_eda, print_report, DEFAULT_TOP_COLUMNS

# --- adjust these ---
JSONL_PATH = Path("data/crops.jsonl")
MAX_ROWS = None  # e.g. 500_000 for a fast sample; None = entire file
PROGRESS_EVERY = None  # status every N rows; set None to silence
TOP_K = 10
TOP_COLUMNS = DEFAULT_TOP_COLUMNS
STRICT_JSON = False

In [2]:
if not JSONL_PATH.is_file():
    raise FileNotFoundError(f"JSONL not found: {JSONL_PATH.resolve()}")

size_b = JSONL_PATH.stat().st_size
print(f"File: {JSONL_PATH.resolve()}")
print(f"Size: {size_b / (1024**3):.2f} GiB ({size_b:,} bytes)")
print(f"Max rows: {MAX_ROWS if MAX_ROWS is not None else 'no limit'}")
print(f"Progress: every {PROGRESS_EVERY:,} rows" if PROGRESS_EVERY else "Progress: disabled")

File: /Users/shashin/Documents/GitHub/INT Biohackathon 2026/data/crops.jsonl
Size: 22.22 GiB (23,861,041,277 bytes)
Max rows: 500000
Progress: every 250,000 rows


In [3]:
print("Starting EDA scan…", flush=True)
t0 = time.perf_counter()

report = run_eda(
    JSONL_PATH,
    max_rows=MAX_ROWS,
    top_k=TOP_K,
    top_columns=tuple(TOP_COLUMNS),
    strict_json=STRICT_JSON,
    progress_every=PROGRESS_EVERY,
)

wall = time.perf_counter() - t0
print(f"\nNotebook wall time (scan + summaries): {wall:,.1f} s", flush=True)

Starting EDA scan…
[eda] rows=250,000  elapsed=11.6s  rate=21,505/s
[eda] rows=500,000  elapsed=26.9s  rate=18,582/s
[eda] finished  rows=500,000  total_time=26.9s  avg_rate=18,581/s

Notebook wall time (scan + summaries): 26.9 s


In [4]:
if "error" in report:
    print("Run failed:", report["error"])
    print("Rows before failure:", report.get("rows_processed", 0))
else:
    print_report(report)

=== JSONL EDA ===
file:     data/crops.jsonl
rows:     500,000
columns:  39

YEAR (parsed as integer in a reasonable range)
  count: 500,000
  min:   1866
  max:   2025

VALUE (numeric only: commas stripped, withheld codes ignored)
  count: 460,481
  min:   -483000.0
  max:   266567647000.0
  mean:  6618253.461188825

Missing / empty by column (sorted by %; top 25 with any empties)
  CONGR_DISTRICT_CODE: 499,994 (99.999%)
  REGION_DESC: 498,893 (99.779%)
  WATERSHED_DESC: 498,810 (99.762%)
  ZIP_5: 462,369 (92.474%)
  WEEK_ENDING: 430,434 (86.087%)
  CV_%: 385,714 (77.143%)
  COUNTY_ANSI: 242,327 (48.465%)
  COUNTY_NAME: 235,146 (47.029%)
  COUNTY_CODE: 234,550 (46.91%)
  ASD_DESC: 211,031 (42.206%)
  ASD_CODE: 210,954 (42.191%)
  STATE_ANSI: 12,923 (2.585%)
  STATE_ALPHA: 1,481 (0.296%)
  STATE_NAME: 1,481 (0.296%)
  VALUE: 1 (0.0%)

Top values (up to 10 per column)
  COMMODITY_DESC
        89,235  WHEAT
        50,048  CORN
        38,820  HAY
        27,851  SOYBEANS
        19,039 

In [5]:
# Optional: inspect the structured report as JSON / dict keys
if "error" not in report:
    print("Keys:", sorted(report.keys()))
    print("Sample top_values keys:", list(report.get("top_values", {}).keys()))

Keys: ['bad_rows', 'column_count', 'columns', 'input', 'json_parse_errors', 'missing', 'rows', 'top_k', 'top_values', 'value_numeric', 'year']
Sample top_values keys: ['COMMODITY_DESC', 'STATE_ALPHA', 'STATISTICCAT_DESC', 'AGG_LEVEL_DESC', 'GROUP_DESC']


## DataFrames by `STATE_ALPHA` and crop **SALES**

The cells below **stream the JSONL again** (separate passes from the EDA summary). For multi‑GiB files this can take a while; progress prints every `PROGRESS_EVERY` rows.

1. **`df_state`** — every distinct `STATE_ALPHA` in the file (including `(empty)`), with row counts and sums of numeric `VALUE` where parseable.
2. **`df_sales_focus`** — `STATISTICCAT_DESC` contains `SALES`, states TX, NE, KS, MT, ND, MN, SD, NC, CO, MO, and (by default) `GROUP_DESC` containing `CROP` (set `CROP_ONLY = False` in that cell for all commodity groups). Aggregated by `GROUP_DESC`, commodity, `SHORT_DESC`, `YEAR`, `AGG_LEVEL_DESC`, units. Includes a per-state roll-up table plus the top detail rows.

Use the same `SCAN_MAX_ROWS` as `MAX_ROWS` in the first cells when you want dataframe passes aligned with the EDA sample.

In [7]:
import json
import time
from collections import Counter, defaultdict

import pandas as pd

from eda_jsonl import parse_value_numeric

# Match EDA: limit rows for testing, or None for full file
SCAN_MAX_ROWS = None  # e.g. 500_000
PROGRESS_EVERY = 500_000


def iter_jsonl_objects(path):
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


state_counts: Counter[str] = Counter()
state_value_sum = defaultdict(float)
state_value_n = defaultdict(int)

n = 0
print("Pass 1: building df_state (all STATE_ALPHA values)…", flush=True)
t0 = time.perf_counter()
for obj in iter_jsonl_objects(JSONL_PATH):
    if not isinstance(obj, dict):
        continue
    n += 1
    if PROGRESS_EVERY and n % PROGRESS_EVERY == 0:
        print(f"  … {n:,} rows", flush=True)
    sa = str(obj.get("STATE_ALPHA") or "").strip()
    if not sa:
        sa = "(empty)"
    state_counts[sa] += 1
    v = parse_value_numeric(str(obj.get("VALUE", "")))
    if v is not None:
        state_value_sum[sa] += v
        state_value_n[sa] += 1
    if SCAN_MAX_ROWS is not None and n >= SCAN_MAX_ROWS:
        break

elapsed = time.perf_counter() - t0
print(f"Done. {n:,} rows in {elapsed:.1f}s ({n / elapsed:,.0f} rows/s)", flush=True)

pairs = state_counts.most_common()
df_state = pd.DataFrame(
    {
        "STATE_ALPHA": [p[0] for p in pairs],
        "row_count": [p[1] for p in pairs],
        "value_numeric_sum": [state_value_sum[p[0]] for p in pairs],
        "value_numeric_rows": [state_value_n[p[0]] for p in pairs],
    }
)
df_state["STATE_ALPHA"] = df_state["STATE_ALPHA"].astype("string")

df_state

Pass 1: building df_state (all STATE_ALPHA values)…
  … 500,000 rows
  … 1,000,000 rows
  … 1,500,000 rows
  … 2,000,000 rows
  … 2,500,000 rows
  … 3,000,000 rows
  … 3,500,000 rows
  … 4,000,000 rows
  … 4,500,000 rows
  … 5,000,000 rows
  … 5,500,000 rows
  … 6,000,000 rows
  … 6,500,000 rows
  … 7,000,000 rows
  … 7,500,000 rows
  … 8,000,000 rows
  … 8,500,000 rows
  … 9,000,000 rows
  … 9,500,000 rows
  … 10,000,000 rows
  … 10,500,000 rows
  … 11,000,000 rows
  … 11,500,000 rows
  … 12,000,000 rows
  … 12,500,000 rows
  … 13,000,000 rows
  … 13,500,000 rows
  … 14,000,000 rows
  … 14,500,000 rows
  … 15,000,000 rows
  … 15,500,000 rows
  … 16,000,000 rows
  … 16,500,000 rows
  … 17,000,000 rows
  … 17,500,000 rows
  … 18,000,000 rows
  … 18,500,000 rows
  … 19,000,000 rows
  … 19,500,000 rows
  … 20,000,000 rows
  … 20,500,000 rows
  … 21,000,000 rows
  … 21,500,000 rows
  … 22,000,000 rows
  … 22,500,000 rows
Done. 22,540,265 rows in 400.9s (56,221 rows/s)


,STATE_ALPHA,row_count,value_numeric_sum,value_numeric_rows
0,TX,1195137,2.231822e+12,1095274
1,NE,869984,3.020755e+12,839732
2,KS,826600,2.313856e+12,788979
3,MT,824019,7.607856e+11,797240
4,ND,815007,2.927134e+12,796188
5,MN,723688,3.054709e+12,679706
6,SD,723283,1.746386e+12,700313
7,NC,660373,1.396962e+12,601776
8,CO,654859,7.856711e+11,620866
9,MO,653841,1.365855e+12,599126


In [8]:
from IPython.display import display

STATES_FOCUS = {"TX", "NE", "KS", "MT", "ND", "MN", "SD", "NC", "CO", "MO"}
# Restrict to crop-related commodity groups (Quick Stats uses GROUP_DESC like FIELD CROPS)
CROP_ONLY = True

# Keys: state, group, commodity, short_desc, year, agg_level, statisticcat, unit
agg = defaultdict(lambda: {"row_count": 0, "value_sum": 0.0, "value_n": 0})

n = 0
matched = 0
print("Pass 2: crop SALES + focus states (detail by commodity / year / geo)…", flush=True)
t0 = time.perf_counter()
for obj in iter_jsonl_objects(JSONL_PATH):
    if not isinstance(obj, dict):
        continue
    n += 1
    if PROGRESS_EVERY and n % PROGRESS_EVERY == 0:
        print(f"  … {n:,} rows scanned, {matched:,} sales rows kept", flush=True)
    sa = str(obj.get("STATE_ALPHA") or "").strip()
    if sa not in STATES_FOCUS:
        continue
    stat_cat = str(obj.get("STATISTICCAT_DESC") or "").upper()
    if "SALES" not in stat_cat:
        continue
    if CROP_ONLY and "CROP" not in str(obj.get("GROUP_DESC") or "").upper():
        continue
    matched += 1
    key = (
        sa,
        str(obj.get("GROUP_DESC") or "").strip(),
        str(obj.get("COMMODITY_DESC") or "").strip(),
        str(obj.get("SHORT_DESC") or "").strip(),
        str(obj.get("YEAR") or "").strip(),
        str(obj.get("AGG_LEVEL_DESC") or "").strip(),
        str(obj.get("STATISTICCAT_DESC") or "").strip(),
        str(obj.get("UNIT_DESC") or "").strip(),
    )
    b = agg[key]
    b["row_count"] += 1
    v = parse_value_numeric(str(obj.get("VALUE", "")))
    if v is not None:
        b["value_sum"] += v
        b["value_n"] += 1
    if SCAN_MAX_ROWS is not None and n >= SCAN_MAX_ROWS:
        break

elapsed = time.perf_counter() - t0
print(f"Done. scanned {n:,} rows; {matched:,} sales rows in focus states; {elapsed:.1f}s", flush=True)

rows_out = []
for k, v in agg.items():
    rows_out.append(
        {
            "STATE_ALPHA": k[0],
            "GROUP_DESC": k[1],
            "COMMODITY_DESC": k[2],
            "SHORT_DESC": k[3],
            "YEAR": k[4],
            "AGG_LEVEL_DESC": k[5],
            "STATISTICCAT_DESC": k[6],
            "UNIT_DESC": k[7],
            "aggregate_row_count": v["row_count"],
            "value_numeric_sum": v["value_sum"],
            "value_numeric_rows": v["value_n"],
        }
    )

df_sales_focus = pd.DataFrame(rows_out)
if not df_sales_focus.empty:
    for col in ("YEAR", "STATE_ALPHA", "GROUP_DESC", "COMMODITY_DESC", "SHORT_DESC"):
        df_sales_focus[col] = df_sales_focus[col].astype("string")
    df_sales_focus = df_sales_focus.sort_values(
        ["STATE_ALPHA", "value_numeric_sum"],
        ascending=[True, False],
    ).reset_index(drop=True)

print("Shape:", df_sales_focus.shape)
if df_sales_focus.empty:
    print("No matching rows (check CROP_ONLY, STATES_FOCUS, or SALES filter).")
else:
    summary_by_state = (
        df_sales_focus.groupby("STATE_ALPHA", as_index=False)
        .agg(
            sales_rows_in_scan=("aggregate_row_count", "sum"),
            value_numeric_sum=("value_numeric_sum", "sum"),
        )
        .sort_values("value_numeric_sum", ascending=False)
    )
    print("Roll-up: crop SALES in focus states (by underlying row counts)")
    display(summary_by_state)
    df_sales_focus.head(40)

Pass 2: crop SALES + focus states (detail by commodity / year / geo)…
  … 500,000 rows scanned, 5,969 sales rows kept
  … 1,000,000 rows scanned, 12,215 sales rows kept
  … 1,500,000 rows scanned, 18,371 sales rows kept
  … 2,000,000 rows scanned, 24,597 sales rows kept
  … 2,500,000 rows scanned, 30,779 sales rows kept
  … 3,000,000 rows scanned, 36,935 sales rows kept
  … 3,500,000 rows scanned, 43,036 sales rows kept
  … 4,000,000 rows scanned, 49,120 sales rows kept
  … 4,500,000 rows scanned, 55,238 sales rows kept
  … 5,000,000 rows scanned, 61,517 sales rows kept
  … 5,500,000 rows scanned, 67,584 sales rows kept
  … 6,000,000 rows scanned, 73,657 sales rows kept
  … 6,500,000 rows scanned, 79,707 sales rows kept
  … 7,000,000 rows scanned, 85,843 sales rows kept
  … 7,500,000 rows scanned, 92,194 sales rows kept
  … 8,000,000 rows scanned, 98,385 sales rows kept
  … 8,500,000 rows scanned, 104,540 sales rows kept
  … 9,000,000 rows scanned, 110,608 sales rows kept
  … 9,500,000

,STATE_ALPHA,sales_rows_in_scan,value_numeric_sum
2,MN,29971,1.135718e+12
7,NE,25060,1.024700e+12
6,ND,18942,7.834262e+11
1,KS,28998,6.305431e+11
8,SD,18602,5.525914e+11
9,TX,58110,5.006693e+11
3,MO,32876,4.992307e+11
5,NC,31081,2.738115e+11
4,MT,15352,1.815288e+11
0,CO,17185,1.783793e+11
